In [0]:
# Databricks notebook source

# ============================================
# PARAMETRO DE ENV VINDO DA PIPELINE / JOB
# ============================================
dbutils.widgets.text("env", "")
p_env = dbutils.widgets.get("env")

env_norm = p_env.strip().lower()
if env_norm in ("hmg", "homol", "hml"):
    env_suffix = "hml"
elif env_norm in ("prod", "prd"):
    env_suffix = "prd"
else:
    env_suffix = "dev"

control_catalog = f"platform_{env_suffix}"

spark.sql(f"""
CREATE OR REPLACE PROCEDURE {control_catalog}.utils.proc_run_silver_scd1
(
  IN p_config_name STRING,
  IN p_layer       STRING,
  IN p_system      STRING,
  IN p_env         STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
  DECLARE v_source STRING;
  DECLARE v_target STRING;

  DECLARE v_src_cat STRING;
  DECLARE v_src_sch STRING;
  DECLARE v_src_tbl STRING;

  DECLARE v_tgt_cat STRING;
  DECLARE v_tgt_sch STRING;
  DECLARE v_tgt_tbl STRING;

  DECLARE v_env STRING;
  DECLARE v_strategy STRING;
  DECLARE v_pk_count INT;
  DECLARE v_pklist STRING;
  DECLARE v_pkcond STRING;
  DECLARE v_updset STRING;
  DECLARE v_inscols STRING;
  DECLARE v_insvals STRING;
  DECLARE merge_sql STRING;

  -- 1) Load config
  CALL {control_catalog}.utils.proc_load_config(
    p_config_name,
    p_layer,
    p_system
  );

  -- 2) Prepare target and sync schema
  CALL {control_catalog}.utils.proc_prepare_target(
    p_config_name,
    p_layer,
    p_system,
    p_env
  );

  CALL {control_catalog}.utils.proc_schema_check(
    p_config_name,
    p_layer,
    p_system,
    p_env
  );

  -- 3) Normalize env
  SET v_env = CASE
    WHEN LOWER(TRIM(p_env)) IN ('hmg', 'homol', 'hml') THEN 'hml'
    WHEN LOWER(TRIM(p_env)) IN ('prod', 'prd') THEN 'prd'
    ELSE LOWER(TRIM(p_env))
  END;

  -- 4) Resolve source and target
  SET v_src_cat = (
    SELECT COALESCE(NULLIF(TRIM(cfg_source.source_catalog), ''), '{control_catalog}')
    FROM cfg
    LIMIT 1
  );

  SET v_src_sch = (
    SELECT NULLIF(TRIM(cfg_source.source_schema), '')
    FROM cfg
    LIMIT 1
  );

  SET v_src_tbl = (
    SELECT NULLIF(TRIM(cfg_source.source_object), '')
    FROM cfg
    LIMIT 1
  );

  SET v_tgt_cat = (
    SELECT COALESCE(NULLIF(TRIM(cfg_target.target_catalog), ''), '{control_catalog}')
    FROM cfg
    LIMIT 1
  );

  SET v_tgt_sch = (
    SELECT NULLIF(TRIM(cfg_target.target_schema), '')
    FROM cfg
    LIMIT 1
  );

  SET v_tgt_tbl = (
    SELECT NULLIF(TRIM(cfg_target.target_object), '')
    FROM cfg
    LIMIT 1
  );

  -- 5) Apply env suffix only when catalog does not already have one
  SET v_src_cat = CASE
    WHEN v_src_cat RLIKE '^(?i).+_(dev|hml|prd)$' THEN v_src_cat
    ELSE CONCAT(v_src_cat, '_', v_env)
  END;

  SET v_tgt_cat = CASE
    WHEN v_tgt_cat RLIKE '^(?i).+_(dev|hml|prd)$' THEN v_tgt_cat
    ELSE CONCAT(v_tgt_cat, '_', v_env)
  END;

  IF v_src_sch IS NULL OR v_src_tbl IS NULL THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_run_silver_scd1: invalid source';
  END IF;

  IF v_tgt_sch IS NULL OR v_tgt_tbl IS NULL THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_run_silver_scd1: invalid target';
  END IF;

  SET v_source = CONCAT('`', v_src_cat, '`.`', v_src_sch, '`.`', v_src_tbl, '`');
  SET v_target = CONCAT('`', v_tgt_cat, '`.`', v_tgt_sch, '`.`', v_tgt_tbl, '`');

  -- 6) Build contract columns from target schema definition
  CREATE OR REPLACE TEMP VIEW tmp_contract_cols AS
  SELECT
    LOWER(e.key) AS col_name_lc,
    e.key AS col_name_raw,
    UPPER(TRIM(e.value)) AS data_type_raw
  FROM cfg
  LATERAL VIEW explode(map_entries(cfg_target.target_schema_definition)) t AS e;

  -- 7) Build PK list from business_keys
  CREATE OR REPLACE TEMP VIEW tmp_pk_cols AS
  SELECT
    LOWER(TRIM(k)) AS pk_lc,
    TRIM(k) AS pk_raw
  FROM cfg
  LATERAL VIEW explode(COALESCE(cfg_target.business_keys, array())) t AS k;

  SET v_pk_count = (
    SELECT COUNT(*)
    FROM tmp_pk_cols
  );

  SET v_strategy = CASE
    WHEN v_pk_count > 0 THEN 'PK'
    ELSE 'HASH'
  END;

  -- 8) PK list for PARTITION BY
  SET v_pklist = (
    SELECT ARRAY_JOIN(
      SORT_ARRAY(COLLECT_LIST(CONCAT('b.`', pk_raw, '`'))),
      ', '
    )
    FROM tmp_pk_cols
  );

  -- 9) PK condition for MERGE ON
  SET v_pkcond = (
    SELECT ARRAY_JOIN(
      SORT_ARRAY(COLLECT_LIST(CONCAT('t.`', pk_raw, '` <=> s.`', pk_raw, '`'))),
      ' AND '
    )
    FROM tmp_pk_cols
  );

  -- 10) Update set for non-PK columns
  CREATE OR REPLACE TEMP VIEW tmp_update_cols AS
  SELECT c.col_name_raw
  FROM tmp_contract_cols c
  LEFT JOIN tmp_pk_cols p
    ON c.col_name_lc = p.pk_lc
  WHERE p.pk_lc IS NULL;

  SET v_updset = (
    SELECT ARRAY_JOIN(
      SORT_ARRAY(COLLECT_LIST(CONCAT('t.`', col_name_raw, '` = s.`', col_name_raw, '`'))),
      ', '
    )
    FROM tmp_update_cols
  );

  -- 11) Insert columns and values
  SET v_inscols = (
    SELECT ARRAY_JOIN(
      SORT_ARRAY(COLLECT_LIST(CONCAT('`', col_name_raw, '`'))),
      ', '
    )
    FROM tmp_contract_cols
  );

  SET v_insvals = (
    SELECT ARRAY_JOIN(
      SORT_ARRAY(COLLECT_LIST(CONCAT('s.`', col_name_raw, '`'))),
      ', '
    )
    FROM tmp_contract_cols
  );

  IF v_inscols IS NULL OR TRIM(v_inscols) = '' THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_run_silver_scd1: insert columns could not be built';
  END IF;

  IF v_insvals IS NULL OR TRIM(v_insvals) = '' THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_run_silver_scd1: insert values could not be built';
  END IF;

  -- 12) Build merge SQL
  IF v_strategy = 'PK' AND v_pk_count > 0 THEN
    SET merge_sql =
      'WITH ranked AS (
         SELECT b.*,
                ROW_NUMBER() OVER (
                  PARTITION BY ' || v_pklist || '
                  ORDER BY b._ingestion_bronze DESC
                ) AS rn
         FROM ' || v_source || ' b
       ),
       latest AS (
         SELECT * FROM ranked WHERE rn = 1
       )
       MERGE INTO ' || v_target || ' AS t
       USING latest AS s
       ON ' || v_pkcond || '
       WHEN MATCHED AND NOT (t.`_hashkey` <=> s.`_hashkey`) THEN
         UPDATE SET ' || v_updset || '
       WHEN NOT MATCHED THEN
         INSERT (' || v_inscols || ') VALUES (' || v_insvals || ')';
  ELSE
    SET merge_sql =
      'WITH ranked AS (
         SELECT b.*,
                ROW_NUMBER() OVER (
                  PARTITION BY b.`_hashkey`
                  ORDER BY b._ingestion_bronze DESC
                ) AS rn
         FROM ' || v_source || ' b
       ),
       latest AS (
         SELECT * FROM ranked WHERE rn = 1
       )
       MERGE INTO ' || v_target || ' AS t
       USING latest AS s
       ON t.`_hashkey` = s.`_hashkey`
       WHEN NOT MATCHED THEN
         INSERT (' || v_inscols || ') VALUES (' || v_insvals || ')';
  END IF;

  EXECUTE IMMEDIATE merge_sql;

  -- 13) Output
  SELECT
    p_config_name AS config_name,
    p_layer AS layer,
    p_system AS system,
    v_strategy AS strategy,
    v_source AS source_table,
    v_target AS target_table,
    'SCD1 merge executed successfully' AS message,
    merge_sql AS executed_sql;
END;
""")